# Why $\Sigma$ Must Be Positive Semi-Definite

A covariance matrix is not just any symmetric table of numbers. Because every portfolio's
variance is $w^\top\Sigma w$, the matrix has to make that number non-negative for **every** $w$.
This notebook shows what that means, how to check it, where broken matrices come from in
practice, and what an optimizer does with one.

## 0. Math link

- [Positive definite matrices](https://github.com/dnlwrthstr/LearningLinearAlgebra/blob/main/hyperskill/05_matrix_decomposition/056_positive_definite_matrices.ipynb):
  definition, eigenvalue test, Cholesky.
- [Spectral decomposition](https://github.com/dnlwrthstr/LearningLinearAlgebra/blob/main/hyperskill/05_matrix_decomposition/055_spectral_decomposition.ipynb):
  $\Sigma = Q\Lambda Q^\top$ for symmetric matrices.
- [Covariance matrix](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/03_probability_distribution_with_multiple_variables/08_covariance_matrix.ipynb)
  and the previous notebook, [01 Portfolio variance](./01_portfolio_variance.ipynb).

## 1. Introduction

The advanced exercise in 04/01 built a "correlation matrix" in which every entry was between
$-1$ and $1$, yet one portfolio came out with variance $-2.4$. Nothing was wrong with any
single number; the **combination** was impossible.

That is what positive semi-definiteness (PSD) rules out. It matters in practice because
non-PSD matrices are produced all the time: by hand-set stress correlations, by estimating
each pair on a different data window, and by having more assets than observations.

After this notebook you can:

- explain why $\Sigma$ must be PSD and why every PSD matrix is a valid covariance matrix;
- test a matrix with eigenvalues or a Cholesky factorization;
- recognize the three usual sources of broken or singular matrices;
- repair a non-PSD correlation matrix with eigenvalue clipping.

## 2. Theory

### The definition, from finance

A symmetric matrix $\Sigma$ is **positive semi-definite** if

$$
w^\top \Sigma\, w \ge 0 \quad \text{for every } w \in \mathbb{R}^n
$$

and **positive definite** (PD) if the inequality is strict for every $w \neq 0$.

For a covariance matrix this is not an assumption, it is forced:
$w^\top\Sigma w = \operatorname{Var}(w^\top R) \ge 0$, for **any** weight vector, including
long-short ones that do not sum to 1.

### The eigenvalue test

A symmetric matrix has the spectral decomposition $\Sigma = Q\Lambda Q^\top$ with orthonormal
eigenvectors $q_k$ (columns of $Q$) and real eigenvalues $\lambda_k$. Substituting $y = Q^\top w$:

$$
w^\top\Sigma\, w = y^\top \Lambda\, y = \sum_{k=1}^n \lambda_k\, y_k^2 = \sum_{k=1}^n \lambda_k\,(q_k^\top w)^2
$$

- All $\lambda_k \ge 0$ $\iff$ PSD. All $\lambda_k > 0$ $\iff$ PD.
- If some $\lambda_k < 0$, the portfolio $w = q_k$ has variance $\lambda_k < 0$.

**Finance reading**: each eigenvector $q_k$ is a portfolio, and its eigenvalue is that
portfolio's variance. The eigenvectors are uncorrelated with each other (module 07/04 calls
them principal components).

### The converse: every PSD matrix is a covariance matrix

If $\Sigma$ is PSD, set $A = Q\Lambda^{1/2}$ (real, since $\lambda_k \ge 0$) and $R = \mu + AZ$ with
$Z$ a vector of independent standard normals. Then

$$
\operatorname{Cov}(R) = A\,\operatorname{Cov}(Z)\,A^\top = Q\Lambda^{1/2}\Lambda^{1/2}Q^\top = \Sigma
$$

So PSD is **exactly** the condition: no more, no less. This is also how correlated returns are
simulated, usually with the Cholesky factor $\Sigma = LL^\top$ in place of $A$ (it exists
for PD matrices).

### Zero eigenvalues: riskless combinations

$\lambda_k = 0$ means the portfolio $q_k$ has **zero** variance: some combination of the assets
is constant. Then $\Sigma$ is singular, $\Sigma^{-1}$ does not exist, and any formula with
$\Sigma^{-1}$ (minimum-variance and tangency portfolios, 04/03 and 04/04) breaks. Typical
causes: a redundant asset (a fund that is a fixed mix of other assets in the matrix) or
fewer observations than assets.

### $2 \times 2$ and $3 \times 3$ correlation matrices

For $C = \begin{pmatrix} 1 & \rho \\ \rho & 1 \end{pmatrix}$ the eigenvalues are $1 \pm \rho$, with
eigenvectors $(1,1)/\sqrt2$ and $(1,-1)/\sqrt2$: the **sum** and the **difference** portfolios of
03/01. PSD $\iff |\rho| \le 1$.

For three assets the pairwise bounds are not enough. Given $\rho_{12}$ and $\rho_{13}$,
the matrix is PSD only if

$$
\rho_{12}\rho_{13} - \sqrt{(1-\rho_{12}^2)(1-\rho_{13}^2)}
\;\le\; \rho_{23} \;\le\;
\rho_{12}\rho_{13} + \sqrt{(1-\rho_{12}^2)(1-\rho_{13}^2)}
$$

which follows from $\det C = 1 + 2\rho_{12}\rho_{13}\rho_{23} - \rho_{12}^2 - \rho_{13}^2 - \rho_{23}^2 \ge 0$.
Correlations constrain each other.

## 3. Worked Examples

### Example 1: the 04/01 matrix

$\rho_{12} = 0.9$, $\rho_{13} = 0.9$, $\rho_{23} = -0.9$.

$$
\det C = 1 + 2(0.9)(0.9)(-0.9) - 3(0.81) = 1 - 1.458 - 2.43 = -2.888 < 0
$$

A negative determinant means an odd number of negative eigenvalues, so the matrix is not PSD.
The admissible range for $\rho_{23}$ is

$$
0.81 \pm \sqrt{0.19 \cdot 0.19} = 0.81 \pm 0.19 \quad\Longrightarrow\quad \rho_{23} \in [0.62,\ 1]
$$

If two assets are each 0.9 correlated with a third, they must be at least 0.62 correlated with
each other.

### Example 2: a fully invested portfolio with negative variance

In correlation units (all variances 1), take $w = (-1,\ 1,\ 1)^\top$, which sums to 1:

$$
w^\top C\, w = 3 + 2\big[(-1)(1)(0.9) + (-1)(1)(0.9) + (1)(1)(-0.9)\big] = 3 - 5.4 = -2.4
$$

### Example 3: a redundant asset

Asset 3 is a fund holding 50% of asset 1 and 50% of asset 2: $R_3 = 0.5R_1 + 0.5R_2$.
Then $w = (0.5,\ 0.5,\ -1)^\top$ has return exactly zero, so $w^\top\Sigma w = 0$: $\Sigma$ is
PSD but **singular**. Nothing is wrong with the data; the matrix just cannot be inverted.

### Example 4: more assets than observations

50 stocks, 36 monthly returns. The sample covariance matrix is built from 36 demeaned
return vectors, which span at most 35 dimensions. So at least $50 - 35 = 15$ eigenvalues are
exactly zero: 15 independent long-short portfolios that **look** riskless in-sample.

## 4. Finance Application

### Where non-PSD matrices come from

- **Hand-set or stressed correlations.** A risk manager sets "crisis correlations" entry by
  entry (all equities to 0.95, one hedge to $-0.8$). Each entry is plausible; the set may not be.
- **Pairwise estimation on different windows.** Assets with different histories (new listings,
  data gaps, different holidays) get each correlation from the overlap of that pair only.
  Each entry comes from a different period and possibly a different market regime.
- **Combining blocks from different sources**, e.g. an equity block from one model and a
  bond block from another, with cross-correlations from a third.

### Where singular matrices come from

- **More assets than observations** ($n > T$): the sample covariance has rank at most $T-1$.
  Common with daily risk on large universes and short histories, or monthly data on 100+ assets.
- **Redundant instruments**: an index and all its constituents, or a fund and its holdings.

### Why it matters: the optimizer finds the hole

A mean-variance optimizer looks for low-variance portfolios. If the matrix has a negative
eigenvalue, it finds a portfolio with **negative** variance and leverages it without limit. If
the matrix is singular because $n > T$, it finds portfolios that are riskless in-sample and
risky out-of-sample. Both produce absurd weights, and neither raises an error by itself.

### Repairs

- **Eigenvalue clipping**: replace negative eigenvalues by a small positive number, rebuild,
  rescale to unit diagonal. Simple, and used below.
- **Nearest correlation matrix** (Higham, 2002): the closest PSD correlation matrix in a
  weighted norm. The standard tool in risk systems.
- **Shrinkage** (module 08/02): blend the sample matrix with a structured one. Fixes
  singularity and reduces estimation error at the same time.

### The usual misreading

"Every correlation is between $-1$ and $1$, so the matrix is fine." The bounds on single
entries are necessary, not sufficient. Check the eigenvalues.

## 5. Python Examples

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

### Two tests: eigenvalues and Cholesky

In [ ]:
def min_eigenvalue(matrix):
    # eigvalsh: for symmetric matrices, returns real eigenvalues in ascending order
    return np.linalg.eigvalsh(matrix)[0]


def is_psd(matrix, tol=1e-10):
    return min_eigenvalue(matrix) >= -tol


def has_cholesky(matrix):
    try:
        np.linalg.cholesky(matrix)
        return True
    except np.linalg.LinAlgError:
        return False


good_corr = np.array([
    [1.0, 0.8, 0.1],
    [0.8, 1.0, 0.0],
    [0.1, 0.0, 1.0],
])   # from 04/01
bad_corr = np.array([
    [1.0, 0.9, 0.9],
    [0.9, 1.0, -0.9],
    [0.9, -0.9, 1.0],
])

for name, C in [("04/01 matrix", good_corr), ("bad matrix", bad_corr)]:
    print(f"{name:12s}: eigenvalues {np.round(np.linalg.eigvalsh(C), 3)}, "
          f"PSD {is_psd(C)}, Cholesky {has_cholesky(C)}, det {np.linalg.det(C):+.3f}")

### Eigenvectors are portfolios, eigenvalues their variances

For the $2 \times 2$ case the eigenvectors are the sum and difference portfolios.

In [ ]:
rho = 0.6
C2 = np.array([[1.0, rho], [rho, 1.0]])
eigvals, eigvecs = np.linalg.eigh(C2)

for lam, q in zip(eigvals, eigvecs.T):
    print(f"eigenvalue {lam:.2f}  eigenvector {np.round(q, 3)}  q'Cq = {q @ C2 @ q:.2f}")

### The negative-variance portfolio, and what an optimizer would do with it

Start from asset 1 alone and move along $d = (-2,\ 1,\ 1)$, which keeps the weights summing
to 1. Along that line the variance is a parabola in the step size $t$ with leading coefficient
$d^\top C d$.

In [ ]:
w_start = np.array([1.0, 0.0, 0.0])
d = np.array([-2.0, 1.0, 1.0])     # sums to 0, so w_start + t d stays fully invested

print(f"d'Cd = {d @ bad_corr @ d:+.2f}  (negative: variance -> -infinity as t grows)")

t_grid = np.linspace(0, 3, 301)
var_path = np.array([(w_start + t * d) @ bad_corr @ (w_start + t * d) for t in t_grid])

w_neg = w_start + 1.0 * d
print(f"t = 1: w = {w_neg}, sum = {w_neg.sum():.0f}, variance = {w_neg @ bad_corr @ w_neg:+.2f}")

NumPy will even try to simulate from it, with only a warning, and the sample covariance of the
draws is **not** the matrix you asked for:

In [ ]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    draws = rng.multivariate_normal(np.zeros(3), bad_corr, size=100_000)
print("warning:", caught[0].message if caught else "none")
print("sample correlation of the draws:")
print(np.round(np.corrcoef(draws, rowvar=False), 2))

### The admissible range of $\rho_{23}$ (Example 1)

In [ ]:
rho_12, rho_13 = 0.9, 0.9
half_width = np.sqrt((1 - rho_12**2) * (1 - rho_13**2))
lo, hi = rho_12 * rho_13 - half_width, rho_12 * rho_13 + half_width
print(f"admissible rho_23: [{lo:.2f}, {hi:.2f}]")

rho_23_grid = np.linspace(-1, 1, 401)
min_eigs = []
for r23 in rho_23_grid:
    C3 = np.array([[1, rho_12, rho_13], [rho_12, 1, r23], [rho_13, r23, 1]])
    min_eigs.append(min_eigenvalue(C3))
min_eigs = np.array(min_eigs)

### A redundant asset (Example 3)

In [ ]:
vols = np.array([0.16, 0.17])
C12 = np.array([[1.0, 0.8], [0.8, 1.0]])
Sigma_12 = np.diag(vols) @ C12 @ np.diag(vols)

# Asset 3 = 0.5 * asset 1 + 0.5 * asset 2, so its row of Sigma is a mix of rows 1 and 2
mix = np.array([0.5, 0.5])
A = np.vstack([np.eye(2), mix])          # R = A @ (R1, R2)
Sigma_red = A @ Sigma_12 @ A.T

w_flat = np.array([0.5, 0.5, -1.0])
print(f"eigenvalues: {np.linalg.eigvalsh(Sigma_red)}")
print(f"variance of (0.5, 0.5, -1): {w_flat @ Sigma_red @ w_flat:.2e}")
print(f"condition number: {np.linalg.cond(Sigma_red):.1e}   (huge: not safely invertible)")

### Pairwise estimation on different windows

Three assets, three market regimes of 250 days each. The data are incomplete, so each pair is
observed together in only one regime:

| | regime 1 | regime 2 | regime 3 |
|---|---|---|---|
| asset 1 | observed | observed | missing |
| asset 2 | observed | missing | observed |
| asset 3 | missing | observed | observed |

In each regime the true correlation matrix is valid. `pandas.DataFrame.corr` uses pairwise
complete observations by default.

In [ ]:
regime_corrs = [
    np.array([[1, 0.9, 0.5], [0.9, 1, 0.5], [0.5, 0.5, 1]]),    # regime 1: assets 1, 2 move together
    np.array([[1, 0.5, 0.9], [0.5, 1, 0.4], [0.9, 0.4, 1]]),    # regime 2: assets 1, 3 move together
    np.array([[1, 0.0, 0.0], [0.0, 1, -0.9], [0.0, -0.9, 1]]),  # regime 3: assets 2, 3 move apart
]
missing = [2, 1, 0]   # asset not observed in each regime (0-based)

blocks = []
for C_regime, miss in zip(regime_corrs, missing):
    assert is_psd(C_regime)
    block = rng.multivariate_normal(np.zeros(3), 0.01**2 * C_regime, size=250)   # daily returns, 1% vol
    block[:, miss] = np.nan
    blocks.append(block)

daily = pd.DataFrame(np.vstack(blocks), columns=["asset 1", "asset 2", "asset 3"])
pairwise_corr = daily.corr()   # pairwise complete observations

print(pairwise_corr.round(2))
print(f"\neigenvalues: {np.round(np.linalg.eigvalsh(pairwise_corr.values), 3)}")

Every regime was internally consistent; the patchwork is not.

### More assets than observations (Example 4)

50 stocks with a simple one-factor structure (every pair correlated 0.3, 25% annual volatility),
36 monthly returns.

In [ ]:
n_assets, n_months, periods_per_year = 50, 36, 12
sigma_stock, rho_stock = 0.25, 0.3

C_true = np.full((n_assets, n_assets), rho_stock)
np.fill_diagonal(C_true, 1.0)
Sigma_true = (sigma_stock**2 / periods_per_year) * C_true      # monthly

monthly_r = rng.multivariate_normal(np.zeros(n_assets), Sigma_true, size=n_months)
Sigma_hat = np.cov(monthly_r, rowvar=False)

eig_hat = np.linalg.eigvalsh(Sigma_hat)
print(f"rank of sample covariance: {np.linalg.matrix_rank(Sigma_hat)}  (at most T - 1 = {n_months - 1})")
print(f"eigenvalues below 1e-12:    {np.sum(eig_hat < 1e-12)}")

Take an eigenvector with zero eigenvalue, scale it to a fully invested portfolio, and compare
its in-sample variance with its true variance.

In [ ]:
eigvals_hat, eigvecs_hat = np.linalg.eigh(Sigma_hat)
null_space = eigvecs_hat[:, eigvals_hat < 1e-12]

# A combination of null-space vectors whose weights sum to 1:
# pick the null vector with the largest weight sum, so the rescaling is mild
sums = null_space.sum(axis=0)
q = null_space[:, np.argmax(np.abs(sums))]
w_fake = q / q.sum()

in_sample_vol = np.sqrt(max(w_fake @ Sigma_hat @ w_fake, 0) * periods_per_year)
true_vol = np.sqrt(w_fake @ Sigma_true @ w_fake * periods_per_year)
print(f"weights sum to {w_fake.sum():.2f}, gross exposure {np.abs(w_fake).sum():.1f}x")
print(f"in-sample annual volatility: {in_sample_vol:.2%}")
print(f"true annual volatility:      {true_vol:.2%}")

### Repair by eigenvalue clipping

In [ ]:
def clip_to_psd_correlation(C, floor=1e-6):
    # Clip eigenvalues at a small positive floor, rebuild, then rescale to unit diagonal
    eigvals, eigvecs = np.linalg.eigh(C)
    eigvals = np.maximum(eigvals, floor)
    C_psd = eigvecs @ np.diag(eigvals) @ eigvecs.T
    d = np.sqrt(np.diag(C_psd))
    return C_psd / np.outer(d, d)


C_repaired = clip_to_psd_correlation(bad_corr)

print("original:")
print(bad_corr)
print("\nrepaired:")
print(np.round(C_repaired, 3))
print(f"\nmin eigenvalue: {min_eigenvalue(bad_corr):+.3f} -> {min_eigenvalue(C_repaired):+.2e}")
print(f"Cholesky works now: {has_cholesky(C_repaired)}")
print(f"largest change in any entry: {np.abs(C_repaired - bad_corr).max():.3f}")

The repair moved every correlation by 0.4, from 0.9 to 0.5: the input was far from valid. A
repair should be small; if it is not, fix the inputs instead.

## 6. Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(rho_23_grid, min_eigs)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].axvspan(lo, hi, color="tab:green", alpha=0.15, label=f"PSD: ρ23 in [{lo:.2f}, {hi:.2f}]")
axes[0].axvline(-0.9, color="tab:red", linestyle="--", label="ρ23 = −0.9 (Example 1)")
axes[0].set_title("Smallest eigenvalue, ρ12 = ρ13 = 0.9")
axes[0].set_xlabel("ρ23")
axes[0].set_ylabel("smallest eigenvalue of C")
axes[0].legend(loc="lower right")

axes[1].plot(t_grid, var_path)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title("Variance along w = (1,0,0) + t·(−2,1,1), bad matrix")
axes[1].set_xlabel("step t (leverage)")
axes[1].set_ylabel("portfolio variance (correlation units)")

for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for T in [36, 120, 600]:
    sample = rng.multivariate_normal(np.zeros(n_assets), Sigma_true, size=T)
    eig = np.sort(np.linalg.eigvalsh(np.cov(sample, rowvar=False)))[::-1] * periods_per_year
    ax.semilogy(np.arange(1, n_assets + 1), np.maximum(eig, 1e-20), marker=".", label=f"T = {T} months")

eig_true = np.sort(np.linalg.eigvalsh(Sigma_true))[::-1] * periods_per_year
ax.semilogy(np.arange(1, n_assets + 1), eig_true, color="black", linestyle="--", label="true Σ")
ax.set_title("Eigenvalues of the sample covariance, 50 assets (annualized)")
ax.set_xlabel("eigenvalue rank")
ax.set_ylabel("eigenvalue (log scale)")
ax.set_ylim(1e-20, 10)
ax.grid(True, alpha=0.3, which="both")
ax.legend()
plt.show()

With $T = 36$ the last 15 eigenvalues collapse to numerical zero (shown at the bottom of the
axis). Even with $T = 120$ or $600$, the small eigenvalues are spread far below the true
value: the sample matrix systematically **underestimates** the risk of its lowest-variance
directions, exactly the ones an optimizer loads up on (module 08).

## 7. Exercises

### Basic

1. For $\rho_{12} = \rho_{13} = 0.5$, compute the admissible range for $\rho_{23}$ by hand. Check
   the endpoints with `min_eigenvalue`.
2. Find the eigenvalues of the $2 \times 2$ covariance matrix with $\sigma_1 = 20\%$,
   $\sigma_2 = 10\%$, $\rho = 0.5$ using `np.linalg.eigvalsh`. Is it PD?

### Intermediate

3. A risk system covers 200 assets and uses 2 years of weekly returns. At least how many
   eigenvalues of the sample covariance are zero? What about 1 year of daily returns?
4. Interpret: an optimizer, fed with a sample covariance from 12 monthly returns on 20 funds,
   reports a fully invested minimum-variance portfolio with 0.3% annual volatility and gross
   exposure of 900%. What is going on, and what would you do?

### Advanced

5. Show that if $\Sigma$ is PD, then $\Sigma^{-1}$ is PD too. (Hint: eigenvalues.) Why does
   this matter for formulas like $w_{\text{mv}} \propto \Sigma^{-1}\mathbf{1}$ in 04/03?

## 8. Solutions / Checks

In [ ]:
# 1.
r12, r13 = 0.5, 0.5
hw = np.sqrt((1 - r12**2) * (1 - r13**2))
lo1, hi1 = r12 * r13 - hw, r12 * r13 + hw
print(f"1. admissible rho_23: [{lo1:.2f}, {hi1:.2f}]")
for r23 in [lo1 - 0.01, lo1, hi1]:
    C3 = np.array([[1, r12, r13], [r12, 1, r23], [r13, r23, 1]])
    print(f"   rho_23 = {r23:+.2f}: min eigenvalue {min_eigenvalue(C3):+.4f}")

# 2.
S2 = np.array([[0.2**2, 0.5 * 0.2 * 0.1], [0.5 * 0.2 * 0.1, 0.1**2]])
print(f"2. eigenvalues {np.round(np.linalg.eigvalsh(S2), 5)}, PD: {min_eigenvalue(S2) > 0}")

# 3. rank <= T - 1
for label, T in [("2 years weekly", 2 * 52), ("1 year daily", 252)]:
    print(f"3. {label:15s}: T = {T}, at least {max(200 - (T - 1), 0)} zero eigenvalues")

**1.** $0.25 \pm \sqrt{0.75 \cdot 0.75} = 0.25 \pm 0.75$, so $\rho_{23} \in [-0.5,\ 1]$. Two assets
that are each 0.5 correlated with a third cannot be more than 0.5 negatively correlated with each
other.

**3.** With 104 weekly observations the rank is at most 103, so at least 97 eigenvalues are zero.
With 252 daily observations and 200 assets, $T - 1 = 251 \ge 200$: no forced zeros, but the
smallest eigenvalues are still badly underestimated (see the last chart).

**4.** With $T = 12$ and $n = 20$, the sample covariance has rank at most 11, so at least 9
independent long-short portfolios have zero in-sample variance. The optimizer combined them
into a leveraged portfolio that is nearly riskless **in-sample** only (Example 4). Fixes:
more data (or higher frequency), fewer assets, a factor model, or shrinkage (module 08), and
constraints on leverage.

**5.** $\Sigma = Q\Lambda Q^\top$ with all $\lambda_k > 0$ gives $\Sigma^{-1} = Q\Lambda^{-1}Q^\top$, whose
eigenvalues $1/\lambda_k$ are all positive. A PD $\Sigma$ has a well-defined, PD inverse, so
minimum-variance formulas give a genuine minimum. If $\Sigma$ has a tiny eigenvalue $\lambda$,
$\Sigma^{-1}$ has a huge one, $1/\lambda$: the formula amplifies exactly the noisy directions,
which is why near-singular matrices produce extreme weights.

## 9. Conclusion

- $w^\top\Sigma w = \operatorname{Var}(w^\top R) \ge 0$ for every $w$: a covariance matrix must be PSD.
  Conversely, every PSD matrix is the covariance of some random vector.
- Test with eigenvalues (all $\ge 0$) or Cholesky (succeeds for PD).
- Eigenvectors are uncorrelated portfolios; eigenvalues are their variances. A negative eigenvalue is a portfolio with negative variance.
- Correlations constrain each other; valid entries do not make a valid matrix.
- Non-PSD: hand-set correlations, pairwise windows, patched blocks. Singular: redundant assets, $n \ge T$.
- Optimizers exploit both. Repair (clipping, nearest correlation matrix, shrinkage) and check before optimizing.

Next: [03 Global minimum-variance portfolio with Lagrange multipliers](./03_global_minimum_variance_portfolio.ipynb).